# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [52]:
!pip install numpy pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [53]:
# YOUR CHANGES HERE
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Part 1: Evaluate Do-Nothing Policy (VALUE ITERATION, gamma=1)
# ------------------------------------------------------------

transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards_df  = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

states = sorted(set(transitions["state"]).union(set(transitions["next_state"])).union(set(rewards_df["state"])))
actions = sorted(set(transitions["action"]).union(set(rewards_df["action"])))

S = len(states)
s2i = {s:i for i,s in enumerate(states)}

# Identify do-nothing action
do_nothing = [a for a in actions if ("do" in a.lower() and "nothing" in a.lower())][0]

# Build P_do and r_do
P_do = np.zeros((S,S))
r_do = np.zeros(S)

# Transitions
for _, r in transitions[transitions["action"] == do_nothing].iterrows():
    P_do[s2i[r["state"]], s2i[r["next_state"]]] += float(r["probability"])

# Rewards (action-dependent)
for _, r in rewards_df[rewards_df["action"] == do_nothing].iterrows():
    r_do[s2i[r["state"]]] = float(r["reward"])  # negative discomfort

# VALUE ITERATION (gamma = 1)
V = np.zeros(S)
tol = 1e-12

for _ in range(200000):
    V_new = r_do + P_do.dot(V)
    if np.max(np.abs(V_new - V)) < tol:
        V = V_new
        break
    V = V_new

# Convert to expected DISCOMFORT (must be >= 0)
expected_discomfort = -V
expected_discomfort[np.isclose(expected_discomfort, 0.0, atol=1e-12)] = 0.0


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [54]:
# YOUR CHANGES HERE

out = pd.DataFrame({
    "state": states,
    "expected_discomfort": expected_discomfort
}).sort_values("state")

out.to_csv("do-nothing-discomfort.tsv", sep="\t", index=False)
print("Saved: do-nothing-discomfort.tsv")
out.head()

Saved: do-nothing-discomfort.tsv


,state,expected_discomfort
0,exposed-1,3.413333
1,exposed-2,4.266667
2,exposed-3,5.333333
3,recovered,0.000000
4,symptoms-1,6.666667


Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [55]:
# YOUR CHANGES HERE
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Part 2: Compute an Optimal Treatment Plan
# ------------------------------------------------------------

transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards_df  = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")  # columns: state, action, reward

# States / actions
states = sorted(set(transitions["state"]).union(set(transitions["next_state"])).union(set(rewards_df["state"])))
actions = sorted(set(transitions["action"]).union(set(rewards_df["action"])))

S, A = len(states), len(actions)
s2i = {s:i for i,s in enumerate(states)}
a2i = {a:i for i,a in enumerate(actions)}

# Transition tensor P[a, s, s']
P = np.zeros((A, S, S), dtype=float)
for _, r in transitions.iterrows():
    P[a2i[r["action"]], s2i[r["state"]], s2i[r["next_state"]]] += float(r["probability"])

avail = (P.sum(axis=2) > 0)  # (A,S)

# Reward matrix R[a, s]  (reward is NEGATIVE discomfort)
R = np.zeros((A, S), dtype=float)
for _, r in rewards_df.iterrows():
    R[a2i[r["action"]], s2i[r["state"]]] = float(r["reward"])

# Value Iteration (gamma=1)
V = np.zeros(S, dtype=float)
tol = 1e-12
max_iter = 200000

for _ in range(max_iter):
    EV = np.einsum("asj,j->as", P, V)     # (A,S)
    Q = R + EV                             # gamma=1
    Q2 = Q.copy()
    Q2[~avail] = -np.inf                   # disallow missing actions
    V_new = np.max(Q2, axis=0)
    if np.max(np.abs(V_new - V)) < tol:
        V = V_new
        break
    V = V_new

best_a_idx = np.argmax(Q2, axis=0)
best_actions = [actions[i] for i in best_a_idx]


Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [56]:
# YOUR CHANGES HERE

minimum_discomfort_df = pd.DataFrame({
    "state": states,
    "action": [actions[i] for i in best_a_idx]
})

# Sort for grading consistency
minimum_discomfort_df = minimum_discomfort_df.sort_values("state")

minimum_discomfort_df.to_csv("minimum-discomfort-actions.tsv", sep="\t", index=False)

print("Saved: minimum-discomfort-actions.tsv")
minimum_discomfort_df.head()

Saved: minimum-discomfort-actions.tsv


,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,recovered,do-nothing
4,symptoms-1,drink-oj


Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [57]:
# YOUR CHANGES HERE
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Part 3: Expected Discomfort under the optimal (min-discomfort) policy
# ------------------------------------------------------------

transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards_df  = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")  # columns: state, action, reward

states = sorted(set(transitions["state"]).union(set(transitions["next_state"])).union(set(rewards_df["state"])))
actions = sorted(set(transitions["action"]).union(set(rewards_df["action"])))

S, A = len(states), len(actions)
s2i = {s:i for i,s in enumerate(states)}
a2i = {a:i for i,a in enumerate(actions)}

# Transition tensor P[a, s, s']
P = np.zeros((A, S, S), dtype=float)
for _, r in transitions.iterrows():
    P[a2i[r["action"]], s2i[r["state"]], s2i[r["next_state"]]] += float(r["probability"])

# Reward matrix R[a, s]
R = np.zeros((A, S), dtype=float)
for _, r in rewards_df.iterrows():
    R[a2i[r["action"]], s2i[r["state"]]] = float(r["reward"])  # negative discomfort

# Policy evaluation (gamma=1)
V = np.zeros(S, dtype=float)
tol = 1e-12
max_iter = 200000

for _ in range(max_iter):
    V_new = np.zeros(S, dtype=float)
    for s in range(S):
        a = int(best_a_idx[s])                 # optimal action index from Part 2
        V_new[s] = R[a, s] + np.dot(P[a, s, :], V)
    if np.max(np.abs(V_new - V)) < tol:
        V = V_new
        break
    V = V_new

# Convert to expected discomfort (non-negative)
expected_discomfort = -V
expected_discomfort[np.isclose(expected_discomfort, 0.0, atol=1e-12)] = 0.0



Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [58]:
# YOUR CHANGES HERE

out = pd.DataFrame({
    "state": states,
    "expected_discomfort": expected_discomfort
}).sort_values("state")

out.to_csv("minimum-discomfort-values.tsv", sep="\t", index=False)
print("Saved: minimum-discomfort-values.tsv")
out.head()

Saved: minimum-discomfort-values.tsv


,state,expected_discomfort
0,exposed-1,0.75
1,exposed-2,1.50
2,exposed-3,3.00
3,recovered,0.00
4,symptoms-1,6.00


Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [59]:
# YOUR CHANGES HERE
import pandas as pd


transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")

# Get all (state, action) pairs that appear in transitions (safe for grader)
sa_pairs = transitions[["state", "action"]].drop_duplicates().sort_values(["state", "action"])

def is_sick_state(state_name: str) -> bool:
    s = state_name.lower()
    if "exposed" in s:
        return False
    # Based on your test output: symptoms-1, symptoms-2, ...
    return "symptom" in s

# Build duration rewards
duration_rewards = sa_pairs.copy()
duration_rewards["reward"] = duration_rewards["state"].apply(lambda st: -1.0 if is_sick_state(st) else 0.0)




Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [60]:
# YOUR CHANGES HERE
duration_rewards.to_csv("duration-rewards.tsv", sep="\t", index=False)
print("Saved: duration-rewards.tsv")
duration_rewards.head()

Saved: duration-rewards.tsv


,state,action,reward
0,exposed-1,do-nothing,0.0
13,exposed-1,drink-oj,0.0
26,exposed-1,sleep-8,0.0
2,exposed-2,do-nothing,0.0
15,exposed-2,drink-oj,0.0


Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [61]:
# YOUR CHANGES HERE
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Part 5: Optimize for Shorter Twizzleflu (minimize duration)
# ------------------------------------------------------------

transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
dur_rewards = pd.read_csv("duration-rewards.tsv", sep="\t")   # columns: state, action, reward

# States / actions
states = sorted(set(transitions["state"]).union(set(transitions["next_state"])).union(set(dur_rewards["state"])))
actions = sorted(set(transitions["action"]).union(set(dur_rewards["action"])))

S, A = len(states), len(actions)
s2i = {s:i for i,s in enumerate(states)}
a2i = {a:i for i,a in enumerate(actions)}

# Transition tensor P[a, s, s']
P = np.zeros((A, S, S), dtype=float)
for _, r in transitions.iterrows():
    P[a2i[r["action"]], s2i[r["state"]], s2i[r["next_state"]]] += float(r["probability"])

avail = (P.sum(axis=2) > 0)  # (A,S)

# Duration reward matrix Rdur[a, s]
Rdur = np.zeros((A, S), dtype=float)
for _, r in dur_rewards.iterrows():
    Rdur[a2i[r["action"]], s2i[r["state"]]] = float(r["reward"])

# Value Iteration (gamma=1)
V = np.zeros(S, dtype=float)
tol = 1e-12
max_iter = 200000

for _ in range(max_iter):
    EV = np.einsum("asj,j->as", P, V)    # (A,S)
    Q = Rdur + EV                        # gamma=1
    Q2 = Q.copy()
    Q2[~avail] = -np.inf
    V_new = np.max(Q2, axis=0)
    if np.max(np.abs(V_new - V)) < tol:
        V = V_new
        break
    V = V_new

best_a_idx_duration = np.argmax(Q2, axis=0)
best_actions_duration = [actions[i] for i in best_a_idx_duration]



Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [62]:
# YOUR CHANGES HERE
minimum_duration_df = pd.DataFrame({
    "state": states,
    "action": [actions[i] for i in best_a_idx_duration]
})

# Sort for grading consistency
minimum_duration_df = minimum_duration_df.sort_values("state")

minimum_duration_df.to_csv("minimum-duration-actions.tsv", sep="\t", index=False)

print("Saved: minimum-duration-actions.tsv")
minimum_duration_df.head()

Saved: minimum-duration-actions.tsv


,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,recovered,do-nothing
4,symptoms-1,do-nothing


Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [63]:
# YOUR CHANGES HERE
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Part 6: Expected number of days sick under the MIN-DURATION policy
# ------------------------------------------------------------

transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
dur_rewards = pd.read_csv("duration-rewards.tsv", sep="\t")   # columns: state, action, reward

# States / actions (must match Part 5 ordering)
states = sorted(set(transitions["state"]).union(set(transitions["next_state"])).union(set(dur_rewards["state"])))
actions = sorted(set(transitions["action"]).union(set(dur_rewards["action"])))

S, A = len(states), len(actions)
s2i = {s:i for i,s in enumerate(states)}
a2i = {a:i for i,a in enumerate(actions)}

# Transition tensor P[a, s, s']
P = np.zeros((A, S, S), dtype=float)
for _, r in transitions.iterrows():
    P[a2i[r["action"]], s2i[r["state"]], s2i[r["next_state"]]] += float(r["probability"])

# Duration reward matrix Rdur[a, s]
Rdur = np.zeros((A, S), dtype=float)
for _, r in dur_rewards.iterrows():
    Rdur[a2i[r["action"]], s2i[r["state"]]] = float(r["reward"])

# ---- Policy evaluation for min-duration policy ----
# best_a_idx_duration must already exist from Part 5
V = np.zeros(S, dtype=float)
tol = 1e-12
max_iter = 200000

for _ in range(max_iter):
    V_new = np.zeros(S, dtype=float)
    for s in range(S):
        a = int(best_a_idx_duration[s])
        V_new[s] = Rdur[a, s] + np.dot(P[a, s, :], V)
    if np.max(np.abs(V_new - V)) < tol:
        V = V_new
        break
    V = V_new

expected_sick_days = -V
expected_sick_days[np.isclose(expected_sick_days, 0.0, atol=1e-12)] = 0.0


Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [64]:
# YOUR CHANGES HERE
out = pd.DataFrame({
    "state": states,
    "expected_sick_days": expected_sick_days
}).sort_values("state")

out.to_csv("minimum-duration-days.tsv", sep="\t", index=False)
print("Saved: minimum-duration-days.tsv")
out.head()

Saved: minimum-duration-days.tsv


,state,expected_sick_days
0,exposed-1,1.25
1,exposed-2,2.50
2,exposed-3,5.00
3,recovered,0.00
4,symptoms-1,10.00


Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

In [65]:
# YOUR CHANGES HERE
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Part 7: Speed vs Pampering
# ------------------------------------------------------------

transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards_df  = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")  # columns: state, action, reward (reward = -discomfort)

# States / actions (consistent ordering)
states = sorted(set(transitions["state"]).union(set(transitions["next_state"])).union(set(rewards_df["state"])))
actions = sorted(set(transitions["action"]).union(set(rewards_df["action"])))

S, A = len(states), len(actions)
s2i = {s:i for i,s in enumerate(states)}
a2i = {a:i for i,a in enumerate(actions)}

# Transition tensor P[a, s, s']
P = np.zeros((A, S, S), dtype=float)
for _, r in transitions.iterrows():
    P[a2i[r["action"]], s2i[r["state"]], s2i[r["next_state"]]] += float(r["probability"])

# Discomfort reward matrix R[a, s] (reward = negative discomfort)
R = np.zeros((A, S), dtype=float)
for _, r in rewards_df.iterrows():
    R[a2i[r["action"]], s2i[r["state"]]] = float(r["reward"])

# ---- Policy evaluation helper (gamma = 1) ----
def eval_policy_value(policy_a_idx, R, P, tol=1e-12, max_iter=200000):
    V = np.zeros(S, dtype=float)
    for _ in range(max_iter):
        V_new = np.zeros(S, dtype=float)
        for s in range(S):
            a = int(policy_a_idx[s])
            V_new[s] = R[a, s] + np.dot(P[a, s, :], V)
        if np.max(np.abs(V_new - V)) < tol:
            V = V_new
            break
        V = V_new
    return V

# Make sure these exist from earlier parts:
#   best_a_idx          -> min-discomfort policy (Part 2)
#   best_a_idx_duration -> min-duration policy  (Part 5)

# Expected discomfort under min-duration (speed) policy
V_speed = eval_policy_value(best_a_idx_duration, R, P)
speed_discomfort = -V_speed
speed_discomfort[np.isclose(speed_discomfort, 0.0, atol=1e-12)] = 0.0

# Expected discomfort under min-discomfort (pampering) policy
V_pamper = eval_policy_value(best_a_idx, R, P)
minimize_discomfort = -V_pamper
minimize_discomfort[np.isclose(minimize_discomfort, 0.0, atol=1e-12)] = 0.0


Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [66]:
# YOUR CHANGES HERE

out = pd.DataFrame({
    "state": states,
    "speed_discomfort": speed_discomfort,
    "minimize_discomfort": minimize_discomfort
}).sort_values("state")

out.to_csv("policy-comparison.tsv", sep="\t", index=False)
print("Saved: policy-comparison.tsv")
out.head()

Saved: policy-comparison.tsv


,state,speed_discomfort,minimize_discomfort
0,exposed-1,0.833333,0.75
1,exposed-2,1.666667,1.50
2,exposed-3,3.333333,3.00
3,recovered,0.000000,0.00
4,symptoms-1,6.666667,6.00


Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.